# 🦕 DINO SDK v1.2.0 - Demonstração Completa

**Data Integration & Operations SDK para Databricks Unity Catalog**

---

## 🎯 **Objetivo deste Notebook**

Este notebook demonstra **todas as funcionalidades** do DINO SDK v1.2.0:
- ✅ Instalação e verificação
- ✅ Criação de schemas com volumes gerenciados 
- ✅ Ingestão de dados com AutoLoader
- ✅ Liquid Clustering automático
- ✅ Testes de validação
- ✅ Troubleshooting e debugging

**📋 Pré-requisitos:**
- Databricks Runtime 13.x+
- Unity Catalog habilitado
- Catálogo `main` disponível
- Permissões para criar schemas e volumes

## 🚀 **1. Instalação do DINO SDK**

In [ ]:
# Instalar DINO SDK v1.2.0
%pip install /Volumes/main/default/system_files/wheels/dino_sdk-1.2.0-py3-none-any.whl --force-reinstall

# Restart Python para garantir que a instalação funcione
dbutils.library.restartPython()

## ✅ **2. Verificação da Instalação**

In [ ]:
# Verificar se tudo foi instalado corretamente
print("🔍 Verificando instalação do DINO SDK...")
print("=" * 50)

try:
    # Importar componentes principais
    from dino_sdk import IngestionEngine, IngestionConfig
    from dino_sdk.schema_manager import ensure_schema_simple
    import dino_sdk
    
    print("✅ Importações bem-sucedidas!")
    print(f"✅ DINO SDK v{dino_sdk.__version__} carregado")
    print(f"✅ Spark {spark.version} disponível")
    print(f"✅ Unity Catalog: {spark.sql('SELECT current_catalog()').collect()[0][0]}")
    
    print("\n🎉 Instalação verificada com sucesso!")
    
except ImportError as e:
    print(f"❌ Erro de importação: {e}")
except Exception as e:
    print(f"❌ Erro inesperado: {e}")

## 🏗️ **3. Criação de Schema com Volumes**

O DINO SDK cria automaticamente **3 volumes gerenciados** por schema:
- **`_checkpoints`**: Para checkpoints do AutoLoader
- **`_schemas`**: Para schemas do AutoLoader
- **`raw`**: Para dados raw/landing

In [ ]:
# Configuração para demonstração
catalog_name = "main"
schema_name = "dino_demo_bronze"

print(f"🏗️ Criando schema: {catalog_name}.{schema_name}")
print("=" * 60)

# Criar schema + volumes automaticamente
result = ensure_schema_simple(spark, catalog_name, schema_name)

# Exibir resultado detalhado
print("📋 Resultado da criação:")
print(f"   • Sucesso: {result['success']}")
print(f"   • Schema criado: {result.get('schema_created', False)}")
print(f"   • Já existia: {result.get('already_exists', False)}")

if 'volumes_created' in result and result['volumes_created']:
    print(f"   • Volumes criados: {result['volumes_created']}")

if 'volumes_existing' in result and result['volumes_existing']:
    print(f"   • Volumes já existentes: {result['volumes_existing']}")

if result['success']:
    print("\n✅ Schema e volumes configurados com sucesso!")
else:
    print("\n❌ Problema na configuração:")
    for error in result.get('errors', []):
        print(f"   • {error}")

In [ ]:
# Verificar volumes criados
print("🔍 Verificando volumes criados:")
print("=" * 40)

# Listar volumes
volumes_df = spark.sql(f"SHOW VOLUMES IN {catalog_name}.{schema_name}")
volumes_df.show(truncate=False)

# Validar se todos os volumes esperados existem
volumes = volumes_df.collect()
expected_volumes = ["_checkpoints", "_schemas", "raw"]
found_volumes = [vol.volume_name for vol in volumes]

print("\n📦 Status dos volumes:")
for expected in expected_volumes:
    if expected in found_volumes:
        print(f"   ✅ Volume {expected} encontrado")
    else:
        print(f"   ❌ Volume {expected} NÃO encontrado")

print(f"\n🎉 Total de volumes criados: {len(found_volumes)}")

## 📝 **4. Preparação de Dados de Teste**

Vamos criar dados simulados de vendas para demonstrar a ingestão.

In [ ]:
# Criar dados de teste realistas
print("📝 Criando dados de teste de vendas...")
print("=" * 45)

import random
from datetime import datetime, timedelta

# Dados simulados de vendas (1000 registros)
products = ["Notebook Dell", "Mouse Logitech", "Teclado Mecânico", "Monitor 24'", "SSD 500GB", "Webcam HD", "Headset Gamer"]
categories = ["Informática", "Periféricos", "Armazenamento", "Audio/Video"]
regions = ["Norte", "Sul", "Leste", "Oeste", "Centro"]
customers = [f"Cliente_{i:04d}" for i in range(1, 101)]  # 100 clientes

# Gerar dados
test_data = []
base_date = datetime(2024, 1, 1)

for i in range(1000):  # 1000 transações
    order_date = base_date + timedelta(days=random.randint(0, 90))  # 90 dias de dados
    test_data.append((
        f"ORD{i+1:06d}",  # order_id
        order_date.strftime("%Y-%m-%d"),  # order_date
        random.choice(products),  # product
        random.choice(categories),  # category 
        random.choice(regions),  # region
        random.choice(customers),  # customer_id
        round(random.uniform(50.0, 2000.0), 2),  # amount
        random.randint(1, 10)  # quantity
    ))

# Definir schema
columns = ["order_id", "order_date", "product", "category", "region", "customer_id", "amount", "quantity"]
test_df = spark.createDataFrame(test_data, columns)

print(f"✅ {len(test_data)} registros de teste criados")
print("\n📋 Amostra dos dados:")
test_df.show(10, truncate=False)

print("\n📊 Estatísticas dos dados:")
test_df.describe(["amount", "quantity"]).show()

In [ ]:
# Salvar dados no volume 'raw'
test_path = f"/Volumes/{catalog_name}/{schema_name}/raw/sales_demo/"

print(f"💾 Salvando dados de teste em: {test_path}")
print("=" * 60)

# Salvar como CSV com header
test_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(test_path)

print("✅ Dados salvos com sucesso!")

# Verificar arquivos criados
files = dbutils.fs.ls(test_path)
csv_files = [f for f in files if f.name.endswith('.csv')]

print(f"\n📁 Arquivos CSV criados: {len(csv_files)}")
for file in csv_files[:3]:  # Mostrar só os primeiros 3
    print(f"   • {file.name} ({file.size} bytes)")

## 🚀 **5. Ingestão com AutoLoader e Liquid Clustering**

Agora vamos usar o **DINO SDK** para fazer a ingestão completa dos dados com todas as funcionalidades avançadas.

In [ ]:
# Configurar ingestão completa
print("⚙️ Configurando ingestão de dados...")
print("=" * 45)

# Configuração completa do DINO SDK
config = IngestionConfig(
    # Dados de origem
    source_path=test_path,
    file_extension="csv",
    
    # Destino Unity Catalog
    catalog_name=catalog_name,
    schema_name=schema_name,
    table_name="sales_bronze",
    
    # Liquid Clustering (otimização automática)
    liquid_clustering=True,
    clustering_columns=["region", "category"],
    
    # AutoLoader com schema evolution
    schema_evolution_mode="rescue",
    rescue_data_column="_rescued_data",
    
    # Processamento em batch (ideal para notebooks)
    type_run="batch",
    
    # Metadados
    table_comment="Tabela de vendas criada pelo DINO SDK v1.2.0",
    add_ingestion_metadata=True
)

print("✅ Configuração criada:")
print(f"   • Origem: {config.source_path}")
print(f"   • Destino: {config.catalog_name}.{config.schema_name}.{config.table_name}")
print(f"   • Clustering: {config.clustering_columns}")
print(f"   • Schema Evolution: {config.schema_evolution_mode}")
print(f"   • Tipo: {config.type_run}")

In [ ]:
# Executar ingestão
print("🚀 Executando ingestão de dados...")
print("=" * 40)

# Criar engine e processar dados
engine = IngestionEngine(config, spark)
result = engine.process_data()

# Exibir resultado detalhado
print("📋 Resultado da ingestão:")
print(f"   • Sucesso: {result['success']}")

if result['success']:
    print("   ✅ Ingestão executada com sucesso!")
    
    # Informações adicionais se disponíveis
    if 'records_processed' in result:
        print(f"   • Registros: {result['records_processed']}")
    if 'files_processed' in result:
        print(f"   • Arquivos: {result['files_processed']}")
    if 'execution_time' in result:
        print(f"   • Tempo: {result['execution_time']}s")
        
else:
    print("   ❌ Erro na ingestão:")
    for error in result.get('errors', []):
        print(f"      • {error}")

## ✅ **6. Validação dos Resultados**

Vamos verificar se os dados foram ingeridos corretamente e se o Liquid Clustering está funcionando.

In [ ]:
# Verificar tabela criada
table_full_name = f"{catalog_name}.{schema_name}.sales_bronze"

print(f"🔍 Validando tabela: {table_full_name}")
print("=" * 60)

try:
    # Carregar dados da tabela
    df_result = spark.table(table_full_name)
    record_count = df_result.count()
    
    print(f"✅ Tabela encontrada!")
    print(f"📊 Total de registros: {record_count}")
    
    # Verificar schema
    print("\n📋 Schema da tabela:")
    df_result.printSchema()
    
    # Mostrar algumas linhas
    print("\n📋 Primeiros 10 registros:")
    df_result.show(10, truncate=False)
    
except Exception as e:
    print(f"❌ Erro ao acessar tabela: {e}")

In [ ]:
# Verificar Liquid Clustering
print("💎 Verificando Liquid Clustering...")
print("=" * 40)

try:
    # Obter informações detalhadas da tabela
    table_details = spark.sql(f"DESCRIBE TABLE EXTENDED {table_full_name}")
    
    # Filtrar informações de clustering
    clustering_info = table_details.filter(
        table_details.col_name.contains("Clustering") |
        table_details.col_name.contains("Cluster") |
        table_details.col_name.contains("Provider")
    )
    
    if clustering_info.count() > 0:
        print("✅ Liquid Clustering configurado:")
        clustering_info.show(truncate=False)
    else:
        print("ℹ️ Informações de clustering não encontradas no DESCRIBE")
        
        # Método alternativo - verificar properties da tabela
        properties = spark.sql(f"SHOW TBLPROPERTIES {table_full_name}")
        delta_properties = properties.filter(
            properties.key.contains("delta") |
            properties.key.contains("cluster")
        )
        
        if delta_properties.count() > 0:
            print("📋 Propriedades Delta/Clustering:")
            delta_properties.show(truncate=False)
        
except Exception as e:
    print(f"⚠️ Erro ao verificar clustering: {e}")
    print("💡 Isso pode ser normal em algumas versões do Databricks")

In [ ]:
# Análise dos dados ingeridos
print("📊 Análise dos dados ingeridos...")
print("=" * 40)

# Estatísticas por região (coluna de clustering)
print("🌍 Distribuição por região:")
df_result.groupBy("region").count().orderBy("count", ascending=False).show()

# Estatísticas por categoria (coluna de clustering)
print("\n📦 Distribuição por categoria:")
df_result.groupBy("category").count().orderBy("count", ascending=False).show()

# Estatísticas de valores
print("\n💰 Estatísticas financeiras:")
df_result.select("amount", "quantity").describe().show()

# Top 10 clientes por valor
print("\n👥 Top 10 clientes por valor total:")
(
    df_result
    .groupBy("customer_id")
    .agg(
        spark_sum("amount").alias("total_amount"),
        count("order_id").alias("total_orders")
    )
    .orderBy("total_amount", ascending=False)
    .show(10)
)

from pyspark.sql.functions import sum as spark_sum, count

## ⚡ **7. Teste de Performance**

Vamos testar a performance das consultas com Liquid Clustering.

In [ ]:
# Teste de performance com consultas filtradas
print("⚡ Testando performance de consultas...")
print("=" * 45)

import time

# Teste 1: Filtro por região (coluna de clustering)
print("🧪 Teste 1: Filtro por região")
start_time = time.time()

result_region = df_result.filter(df_result.region == "Norte").count()

end_time = time.time()
print(f"   • Registros encontrados: {result_region}")
print(f"   • Tempo de execução: {end_time - start_time:.3f}s")

# Teste 2: Filtro por categoria (coluna de clustering)
print("\n🧪 Teste 2: Filtro por categoria")
start_time = time.time()

result_category = df_result.filter(df_result.category == "Informática").count()

end_time = time.time()
print(f"   • Registros encontrados: {result_category}")
print(f"   • Tempo de execução: {end_time - start_time:.3f}s")

# Teste 3: Filtro combinado (região + categoria)
print("\n🧪 Teste 3: Filtro combinado")
start_time = time.time()

result_combined = (
    df_result
    .filter((df_result.region == "Norte") & (df_result.category == "Informática"))
    .count()
)

end_time = time.time()
print(f"   • Registros encontrados: {result_combined}")
print(f"   • Tempo de execução: {end_time - start_time:.3f}s")

print("\n✅ Testes de performance concluídos!")
print("💡 Com Liquid Clustering, filtros por 'region' e 'category' são otimizados")

## 🔧 **8. Troubleshooting e Debug**

Ferramentas para debug e resolução de problemas.

In [ ]:
# Informações de debug
print("🔧 Informações de Debug")
print("=" * 30)

# Verificar catálogos disponíveis
print("📚 Catálogos disponíveis:")
spark.sql("SHOW CATALOGS").show()

# Verificar schemas no catálogo
print(f"\n🗂️ Schemas em {catalog_name}:")
spark.sql(f"SHOW SCHEMAS IN {catalog_name}").show()

# Verificar tabelas no schema
print(f"\n📋 Tabelas em {catalog_name}.{schema_name}:")
spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name}").show()

# Informações do usuário atual
print("\n👤 Informações do usuário:")
current_user = spark.sql("SELECT current_user() as user").collect()[0]['user']
print(f"   • Usuário atual: {current_user}")

# Configurações do Spark relevantes
print("\n⚙️ Configurações Spark relevantes:")
print(f"   • spark.sql.adaptive.enabled: {spark.conf.get('spark.sql.adaptive.enabled', 'N/A')}")
print(f"   • spark.databricks.delta.autoCompact.enabled: {spark.conf.get('spark.databricks.delta.autoCompact.enabled', 'N/A')}")

In [ ]:
# Health check completo
print("🏥 Health Check do DINO SDK")
print("=" * 35)

health_status = {
    "sdk_imported": False,
    "spark_available": False,
    "unity_catalog": False,
    "schema_exists": False,
    "volumes_exist": False,
    "table_exists": False,
    "data_accessible": False
}

# Verificar importação do SDK
try:
    from dino_sdk import __version__
    health_status["sdk_imported"] = True
    print(f"✅ DINO SDK v{__version__} importado")
except:
    print("❌ Erro na importação do DINO SDK")

# Verificar Spark
try:
    spark.version
    health_status["spark_available"] = True
    print(f"✅ Spark {spark.version} disponível")
except:
    print("❌ Spark não disponível")

# Verificar Unity Catalog
try:
    current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
    health_status["unity_catalog"] = True
    print(f"✅ Unity Catalog ativo: {current_catalog}")
except:
    print("❌ Unity Catalog não disponível")

# Verificar schema
try:
    spark.sql(f"USE {catalog_name}.{schema_name}")
    health_status["schema_exists"] = True
    print(f"✅ Schema {catalog_name}.{schema_name} acessível")
except:
    print(f"❌ Schema {catalog_name}.{schema_name} não acessível")

# Verificar volumes
try:
    volumes = spark.sql(f"SHOW VOLUMES IN {catalog_name}.{schema_name}").count()
    if volumes >= 3:
        health_status["volumes_exist"] = True
        print(f"✅ {volumes} volumes encontrados")
    else:
        print(f"⚠️ Apenas {volumes} volumes encontrados (esperado: 3+)")
except:
    print("❌ Erro ao verificar volumes")

# Verificar tabela
try:
    spark.table(table_full_name)
    health_status["table_exists"] = True
    print(f"✅ Tabela {table_full_name} existe")
except:
    print(f"❌ Tabela {table_full_name} não encontrada")

# Verificar acesso aos dados
try:
    count = spark.table(table_full_name).count()
    health_status["data_accessible"] = True
    print(f"✅ Dados acessíveis: {count} registros")
except:
    print("❌ Erro ao acessar dados")

# Resumo do health check
healthy_components = sum(health_status.values())
total_components = len(health_status)

print(f"\n📊 Status geral: {healthy_components}/{total_components} componentes saudáveis")

if healthy_components == total_components:
    print("🎉 Todos os sistemas funcionando perfeitamente!")
elif healthy_components >= total_components * 0.7:
    print("⚠️ Sistema funcionando com algumas limitações")
else:
    print("❌ Múltiplos problemas detectados - verificar configuração")

## 🧹 **9. Limpeza (Opcional)**

**⚠️ CUIDADO:** Esta seção remove todos os dados criados na demonstração. Execute apenas se quiser limpar o ambiente.

In [ ]:
# CONFIGURAÇÃO DE SEGURANÇA
# Mude para True apenas se quiser executar a limpeza
EXECUTE_CLEANUP = False

if EXECUTE_CLEANUP:
    print("🧹 Executando limpeza da demonstração...")
    print("=" * 45)
    
    try:
        # Remover tabela
        print(f"🗑️ Removendo tabela {table_full_name}...")
        spark.sql(f"DROP TABLE IF EXISTS {table_full_name}")
        print("   ✅ Tabela removida")
        
        # Remover schema e volumes (CASCADE remove tudo)
        print(f"🗑️ Removendo schema {catalog_name}.{schema_name}...")
        spark.sql(f"DROP SCHEMA IF EXISTS {catalog_name}.{schema_name} CASCADE")
        print("   ✅ Schema e volumes removidos")
        
        print("\n🎉 Limpeza concluída com sucesso!")
        
    except Exception as e:
        print(f"❌ Erro durante limpeza: {e}")
        
else:
    print("💡 Para executar limpeza, mude EXECUTE_CLEANUP = True")
    print("⚠️ CUIDADO: Isso removerá todos os dados criados nesta demonstração")
    print(f"   • Tabela: {table_full_name}")
    print(f"   • Schema: {catalog_name}.{schema_name}")
    print(f"   • Volumes: _checkpoints, _schemas, raw")

## 🎉 **Conclusão**

### **✅ Funcionalidades Demonstradas**

Neste notebook demonstramos **todas as funcionalidades principais** do DINO SDK v1.2.0:

1. **✅ Instalação e Verificação**
   - Instalação via wheel
   - Verificação de componentes
   - Validação do ambiente

2. **✅ Schema e Volumes Gerenciados**
   - Criação automática de schema
   - 3 volumes gerenciados por schema
   - Operações idempotentes

3. **✅ Ingestão de Dados Avançada**
   - AutoLoader com schema evolution
   - Liquid Clustering automático
   - Processamento batch e streaming
   - Metadados de ingestão

4. **✅ Validação e Performance**
   - Verificação de dados ingeridos
   - Testes de performance com clustering
   - Análise estatística dos dados

5. **✅ Troubleshooting e Debug**
   - Health check completo
   - Informações de debug
   - Resolução de problemas

### **🦕 DINO SDK v1.2.0 - Pronto para Produção!**

O DINO SDK oferece uma **solução completa e enterprise** para ingestão de dados no Databricks Unity Catalog com:

- 🔄 **AutoLoader integrado** para ingestão incremental
- 💎 **Liquid Clustering** para performance otimizada
- 🏛️ **Unity Catalog** para governança completa
- 📦 **Volumes gerenciados** para organização automática
- ⚡ **CLI integrada** para facilidade de uso
- 🧪 **Testes abrangentes** para confiabilidade

### **📚 Próximos Passos**

- Consulte o **README.md** para documentação completa
- Explore a pasta **examples/** para casos de uso específicos
- Execute **testes unitários** na pasta tests/
- Use **CLI dino-config** para configurações avançadas

---

**🎯 Desenvolvido para simplificar sua ingestão de dados no Databricks!**